# Panel Político (test)
**Date**: 310525

In [1]:
import os
from dotenv import load_dotenv
from openai import OpenAI
import pandas as pd
from tqdm import tqdm
from itertools import combinations

In [2]:
# Cargar las variables de entorno desde el archivo .env
load_dotenv()

# Obtener la clave de API desde la variable de entorno
api_key = os.getenv("OPENAI_API_KEY")

# Verificar que la clave de API esté disponible
if not api_key:
    raise ValueError("La clave de API de OpenAI no está configurada en el archivo .env.")

# Inicializar el cliente de OpenAI con la clave de API
client = OpenAI(api_key=api_key)

In [3]:
# Contexto para el modelo de lenguaje
with open("contexto_política/contexto_político.txt", "r", encoding="utf-8") as f:
    contexto_político = f.read()

with open("contexto_política/perfiles_políticos.txt", "r", encoding="utf-8") as f:
    perfiles_políticos = f.read()

#print(perfiles_políticos)

In [4]:
def ask_openai(prompt: str) -> str:
    response = client.chat.completions.create(
        model="gpt-4o",
        messages=[
            {"role": "system", "content": f'Considera lo siguiente como contexto de la política peruana: {contexto_político}. Considera lo siguiente como contexto de los candidatos: {perfiles_políticos}'},     
            {"role": "user", "content": f'Basado en el contexto que te tienes, contesta lo siguiente: {prompt}'}
        ]
    )
    return response.choices[0].message.content.strip()

In [5]:
# Prueba de la contextualización
ask_openai("Con qué contenidos fuiste contextualizado?")

'Fui contextualizado con el panorama político peruano rumbo a las Elecciones Generales de 2026, que incluye una fragmentación extrema de partidos, la necesidad de alianzas electorales para mejorar la gobernabilidad, y el rol creciente de los movimientos regionales. También se discutió la posibilidad de la figura de un outsider presidencial, el perfil ideal para el próximo presidente según diversos actores, los riesgos para la gobernabilidad ante la alta fragmentación política, y propuestas para mejorar la descentralización y la calidad del voto. Adicionalmente, se me proporcionó información detallada sobre varios candidatos potenciales para las elecciones de 2026, incluyendo sus perfiles, partidos políticos, ideologías y políticas en diversas áreas.'

## Hacer simulación de pareados considerando el contexto de cada persona

In [6]:
df_test = pd.read_csv("contexto_política/base_sintética.csv")
df_test = df_test.sample(n=30, random_state=42)
print(f"Total de preguntas: {df_test.shape}")
df_test

Total de preguntas: (30, 6)


,sexo,edad,zona,region,preferencia_presidencial,opinion_gobierno
521,M,40-70,urbano,Interior del País,César Acuña,Desaprueba
737,M,25-39,urbano,Interior del País,Blanco/Nulo,Desaprueba
740,H,40-70,urbano,Interior del País,Carlos Álvarez,Desaprueba
660,H,40-70,urbano,Interior del País,No precisa,Desaprueba
411,H,25-39,urbano,Lima - Callao,Carlos Álvarez,Desaprueba
678,M,40-70,urbano,Interior del País,Martín Vizcarra,Desaprueba
626,M,18-24,urbano,Lima - Callao,Martín Vizcarra,Desaprueba
513,H,25-39,urbano,Lima - Callao,Alfonso López Chau,Desaprueba
859,M,40-70,urbano,Interior del País,Carlos Álvarez,Desaprueba
136,M,25-39,urbano,Interior del País,Keiko Fujimori,Desaprueba


In [7]:
# Crea carpeta de salida si no existe
os.makedirs("contexto_política", exist_ok=True)

# -----------------------------
# Construye el prompt personalizado para un pareado de candidatos
def construir_prompt_pareado(fila, cand1, cand2, contexto_político, perfiles_políticos):
    perfil = (
        f"Sexo: {fila['sexo']}, Edad: {fila['edad']}, Zona: {fila['zona']}, Región: {fila['region']}, "
        f"Preferencia previa: {fila['preferencia_presidencial']}, Opinión del gobierno: {fila['opinion_gobierno']}."
    )
    
    return (
        f"Perfil del votante: {perfil}\n\n"
        f"Contexto político: {contexto_político}\n"
        f"Perfiles de candidatos: {perfiles_políticos}\n\n"
        f"Si esta persona solo pudiera votar por uno de los siguientes dos candidatos: {cand1} o {cand2}, "
        f"¿por quién votaría?\n\n"
        f"Responde SOLO con el nombre del candidato elegido, sin explicaciones."
    )

# -----------------------------
# Función principal para simular todas las elecciones pareadas
def simular_elecciones_pareadas(df, contexto_político, perfiles_políticos, lista_candidatos):
    # Generar todas las combinaciones únicas de 2 candidatos
    pares = list(combinations(lista_candidatos, 2))
    
    # Recorremos cada par y simulamos los votos fila por fila
    for cand1, cand2 in tqdm(pares, desc="Simulando pareados"):
        nombre_columna = f"{cand1.replace(' ', '')}_vs_{cand2.replace(' ', '')}"
        resultados = []

        for _, fila in df.iterrows():
            try:
                prompt = construir_prompt_pareado(fila, cand1, cand2, contexto_político, perfiles_políticos)
                respuesta = ask_openai(prompt)
            except Exception as e:
                respuesta = f"Error: {e}"

            resultados.append(respuesta.strip())

        df[nombre_columna] = resultados
    
    return df

In [8]:
# Lista de candidatos
"""
lista_candidatos = [
    'Martín Vizcarra', 'Keiko Fujimori', 'Carlos Álvarez',
    'Rafael López Aliaga', 'Hernando De Soto', 'Alfonso López Chau',
    'Antauro Humala', 'Philip Butters', 'Verónika Mendoza',
    'César Acuña', 'Fernando Olivera'
]
"""

lista_candidatos = [
    'Martín Vizcarra', 'Keiko Fujimori', 'Carlos Álvarez'
]

# Ejecutar la simulación de pareados
df_resultado = simular_elecciones_pareadas(df_test, contexto_político, perfiles_políticos, lista_candidatos)

# Guardar los resultados
#df_resultado.to_csv("contexto_política/simulaciones_pareadas.csv", index=False)


Simulando pareados: 100%|██████████| 3/3 [01:48<00:00, 36.02s/it]


In [15]:
df_resultado.head()

,sexo,edad,zona,region,preferencia_presidencial,opinion_gobierno,MartínVizcarra_vs_KeikoFujimori,MartínVizcarra_vs_CarlosÁlvarez,KeikoFujimori_vs_CarlosÁlvarez
521,M,40-70,urbano,Interior del País,César Acuña,Desaprueba,César Acuña,Martín Vizcarra,Keiko Fujimori
737,M,25-39,urbano,Interior del País,Blanco/Nulo,Desaprueba,Martín Vizcarra,Martín Vizcarra,Carlos Álvarez
740,H,40-70,urbano,Interior del País,Carlos Álvarez,Desaprueba,Keiko Fujimori,Carlos Álvarez,Carlos Álvarez
660,H,40-70,urbano,Interior del País,No precisa,Desaprueba,Keiko Fujimori,Martín Vizcarra,Carlos Álvarez
411,H,25-39,urbano,Lima - Callao,Carlos Álvarez,Desaprueba,Keiko Fujimori,Carlos Álvarez,Carlos Álvarez


In [16]:
df_test.MartínVizcarra_vs_CarlosÁlvarez.value_counts(normalize=True)

MartínVizcarra_vs_CarlosÁlvarez
Martín Vizcarra    0.533333
Carlos Álvarez     0.466667
Name: proportion, dtype: float64

**Next Steps**:
- Afinar archivo de auditoria para esta versión

In [10]:
"""
# Version 1
# Tu pregunta base
pregunta_base = "Si tuvieras que votar por Martín Vizcarra o Keiko Fujimori, ¿por quién lo harías?"

# Función auxiliar que genera el prompt por fila
def construir_prompt(fila, contexto_político, perfiles_políticos):
    perfil_votante = (
        f"Sexo: {fila['sexo']}, Edad: {fila['edad']}, Zona: {fila['zona']}, Región: {fila['region']}, "
        f"Preferencia presidencial previa: {fila['preferencia_presidencial']}, "
        f"Opinión del gobierno: {fila['opinion_gobierno']}."
    )

    prompt = (
        f"Dado el siguiente perfil de votante: {perfil_votante}\n\n"
        f"Y considerando el contexto político: {contexto_político}\n"
        f"Y los perfiles de los candidatos: {perfiles_políticos}\n\n"
        f"Responde con una sola palabra: ¿por quién votaría esta persona si solo puede elegir entre "
        f"Martín Vizcarra o Keiko Fujimori?\n\n"
        f"Solo responde con el nombre del candidato elegido, sin explicaciones."
    )
    
    return prompt

# Función que aplica ask_openai() a cada fila
def simular_respuestas(df, contexto_político, perfiles_políticos):
    respuestas = []
    
    for _, fila in tqdm(df.iterrows(), total=len(df)):
        prompt = construir_prompt(fila, contexto_político, perfiles_políticos)
        try:
            respuesta = ask_openai(prompt)
        except Exception as e:
            respuesta = f"Error: {e}"
        respuestas.append(respuesta)
    
    df["respuesta_simulada"] = respuestas
    return df

# Define tus textos antes de usar
contexto_político = "Texto sobre el contexto político peruano..."
perfiles_políticos = "Perfiles detallados de los candidatos..."

# Simula
df_resultado = simular_respuestas(df_test, contexto_político, perfiles_políticos)
df_resultado
# Guardar en CSV si lo deseas
#df_resultado.to_csv("respuestas_simuladas.csv", index=False)

"""

'\n# Version 1\n# Tu pregunta base\npregunta_base = "Si tuvieras que votar por Martín Vizcarra o Keiko Fujimori, ¿por quién lo harías?"\n\n# Función auxiliar que genera el prompt por fila\ndef construir_prompt(fila, contexto_político, perfiles_políticos):\n    perfil_votante = (\n        f"Sexo: {fila[\'sexo\']}, Edad: {fila[\'edad\']}, Zona: {fila[\'zona\']}, Región: {fila[\'region\']}, "\n        f"Preferencia presidencial previa: {fila[\'preferencia_presidencial\']}, "\n        f"Opinión del gobierno: {fila[\'opinion_gobierno\']}."\n    )\n\n    prompt = (\n        f"Dado el siguiente perfil de votante: {perfil_votante}\n\n"\n        f"Y considerando el contexto político: {contexto_político}\n"\n        f"Y los perfiles de los candidatos: {perfiles_políticos}\n\n"\n        f"Responde con una sola palabra: ¿por quién votaría esta persona si solo puede elegir entre "\n        f"Martín Vizcarra o Keiko Fujimori?\n\n"\n        f"Solo responde con el nombre del candidato elegido, sin exp

In [11]:
"""
# Versión 2
# Crear la carpeta de salida si no existe
os.makedirs("contexto_política", exist_ok=True)

# -----------------------------
# Construir el prompt para que GPT devuelva SOLO el nombre del candidato elegido
def construir_prompt_voto(fila, contexto_político, perfiles_políticos):
    # Construir una cadena con los datos del votante
    perfil = (
        f"Sexo: {fila['sexo']}, Edad: {fila['edad']}, Zona: {fila['zona']}, Región: {fila['region']}, "
        f"Preferencia previa: {fila['preferencia_presidencial']}, Opinión del gobierno: {fila['opinion_gobierno']}."
    )
    
    # Armar el prompt incluyendo contexto, perfiles y datos del votante
    return (
        f"Dado el siguiente perfil de votante: {perfil}\n\n"
        f"Y considerando el contexto político: {contexto_político}\n"
        f"Y los perfiles de los candidatos: {perfiles_políticos}\n\n"
        f"¿Por quién votaría esta persona si solo puede elegir entre Martín Vizcarra o Keiko Fujimori?\n"
        f"Responde SOLO con el nombre del candidato, sin explicaciones."
    )

# -----------------------------
# Construir el prompt para solicitar una explicación corta sobre el voto
def construir_prompt_explicacion(fila, voto, contexto_político, perfiles_políticos):
    # Igual que antes, pero ahora pedimos una razón del voto emitido
    perfil = (
        f"Sexo: {fila['sexo']}, Edad: {fila['edad']}, Zona: {fila['zona']}, Región: {fila['region']}, "
        f"Preferencia previa: {fila['preferencia_presidencial']}, Opinión del gobierno: {fila['opinion_gobierno']}."
    )
    
    return (
        f"Con base en el siguiente perfil de votante: {perfil}\n\n"
        f"Y el contexto político: {contexto_político}\n"
        f"Y los perfiles de los candidatos: {perfiles_políticos}\n\n"
        f"La persona votó por {voto}. Explica brevemente y en una sola oración por qué esa persona tomaría esa decisión."
    )

# -----------------------------
# Función principal que recorre todo el DataFrame de votantes y genera:
# - voto simulado
# - explicación
# - un archivo de auditoría en .csv
def simular_y_auditar(df, contexto_político, perfiles_políticos):
    votos = []       # Lista para almacenar por quién vota cada fila
    auditoria = []   # Lista para almacenar los detalles explicativos

    # Recorremos cada fila del DataFrame usando tqdm para mostrar barra de progreso
    for _, fila in tqdm(df.iterrows(), total=len(df)):
        # Paso 1: construir y enviar prompt para el voto
        prompt_voto = construir_prompt_voto(fila, contexto_político, perfiles_políticos)
        try:
            voto = ask_openai(prompt_voto)  # Llama a la API para obtener el voto simulado
        except Exception as e:
            voto = f"Error: {e}"  # En caso de error, guarda el error como resultado

        votos.append(voto)  # Guardamos el voto para agregarlo al DataFrame original

        # Paso 2: construir y enviar prompt para la explicación del voto
        prompt_exp = construir_prompt_explicacion(fila, voto, contexto_político, perfiles_políticos)
        try:
            explicacion = ask_openai(prompt_exp)  # Llama a la API para obtener la razón del voto
        except Exception as e:
            explicacion = f"Error: {e}"

        # Guardamos toda la información en la auditoría (se guardará en un archivo externo)
        auditoria.append({
            "sexo": fila["sexo"],
            "edad": fila["edad"],
            "zona": fila["zona"],
            "region": fila["region"],
            "preferencia_presidencial": fila["preferencia_presidencial"],
            "opinion_gobierno": fila["opinion_gobierno"],
            "voto": voto,
            "razon": explicacion
        })

    # Agregamos los votos simulados al DataFrame original como nueva columna
    df["voto_simulado"] = votos

    # Convertimos la lista de auditoría en un DataFrame y lo guardamos como CSV
    auditoria_df = pd.DataFrame(auditoria)
    auditoria_df.to_csv("contexto_política/auditoria.csv", index=False)

    # Devolvemos el DataFrame original con votos incluidos
    return df

df_resultado = simular_y_auditar(df_test, contexto_político, perfiles_políticos)
"""

'\n# Versión 2\n# Crear la carpeta de salida si no existe\nos.makedirs("contexto_política", exist_ok=True)\n\n# -----------------------------\n# Construir el prompt para que GPT devuelva SOLO el nombre del candidato elegido\ndef construir_prompt_voto(fila, contexto_político, perfiles_políticos):\n    # Construir una cadena con los datos del votante\n    perfil = (\n        f"Sexo: {fila[\'sexo\']}, Edad: {fila[\'edad\']}, Zona: {fila[\'zona\']}, Región: {fila[\'region\']}, "\n        f"Preferencia previa: {fila[\'preferencia_presidencial\']}, Opinión del gobierno: {fila[\'opinion_gobierno\']}."\n    )\n\n    # Armar el prompt incluyendo contexto, perfiles y datos del votante\n    return (\n        f"Dado el siguiente perfil de votante: {perfil}\n\n"\n        f"Y considerando el contexto político: {contexto_político}\n"\n        f"Y los perfiles de los candidatos: {perfiles_políticos}\n\n"\n        f"¿Por quién votaría esta persona si solo puede elegir entre Martín Vizcarra o Keiko Fuji